# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets with their @id and fields
print("Available record sets:")
for record_set in metadata.record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List the @ids of the record sets
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Detected record set @ids: ", record_set_ids)

# Load data from all record sets into dataframes
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
    print(dataframes[record_set_id].head(), "\n")
# For further analysis, select the main record set (usually the first or with most fields)
target_record_set_id = record_set_ids[0]


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations include removing outliers, transforming data distributions, and grouping data by key attributes to prepare for further analysis.

In [ ]:
df = dataframes[target_record_set_id]

# Find a numeric field for analysis (e.g., 'peptide_count' or a mass/abundance field)
numeric_fields = []
for field in metadata.record_sets[0].fields:
    # Check for typical numeric types
    if field.data_type in ("Float", "Integer", "Number"):
        numeric_fields.append(field.id)
print(f"Numeric fields detected in record set {target_record_set_id}: {numeric_fields}")

# Let's pick the first available numeric field for filtering and normalization
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")

    # Show statistics and filter
    threshold = df[numeric_field_id].mean() if numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id in df.columns else df.copy()
    print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    if numeric_field_id in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Group by a categorical field if available (e.g., sample name or protein description)
    group_field_id = None
    # Look for a non-numeric, non-id text field
    for field in metadata.record_sets[0].fields:
        if field.data_type == "Text" and field.id != numeric_field_id:
            group_field_id = field.id
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Distribution of selected numeric field
if numeric_fields and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Example: If a grouping field is available, show boxplot
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10, 6))
    # Only show top 10 groups by size
    top_groups = df[group_field_id].value_counts().nlargest(10).index
    sns.boxplot(
        data=df[df[group_field_id].isin(top_groups)],
        x=group_field_id, y=numeric_field_id
    )
    plt.title(f"{numeric_field_id} grouped by {group_field_id} (top 10)")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR<sup>2</sup> mass spectrometry dataset by loading its Croissant schema, reviewing record sets and fields by their `@id`, extracting tabular data, conducting filtering and normalization based on numeric columns, and visualizing the data distributions. This approach makes the dataset more accessible for downstream analysis and machine learning applications.

You can further customize the EDA by referencing any field, column, or record set using its `@id`, making analyses reproducible and robust to schema changes.